# Vodafone UK Coverage Gap: Stage 1 Data Collection

## 0. Setup

In [11]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install geopandas pyproj pyarrow beautifulsoup4

import os
from pathlib import Path
OUT = Path('/content/drive/MyDrive/vodafone-project/data'); OUT.mkdir(parents=True, exist_ok=True)
RAW = Path('/content/raw'); RAW.mkdir(exist_ok=True)
print('Outputs ->', OUT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Outputs -> /content/drive/MyDrive/vodafone-project/data


## 1. Helper library


In [12]:
%%writefile /content/collect_lib.py

from __future__ import annotations

import io
import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import requests

UK_MNC = {  # MCC 234
    2: "O2", 10: "O2", 11: "O2",
    15: "Vodafone",
    20: "Three",
    30: "EE", 31: "EE", 32: "EE", 33: "EE", 34: "EE",
}
OPS = ["EE", "O2", "Three", "Vodafone"]
RSRP_OUTDOOR_DBM = -105
RSRP_INDOOR_DBM = -95
GRID_M = 1000


def scrape_links(page_url: str) -> pd.DataFrame:
    from bs4 import BeautifulSoup
    html = requests.get(page_url, timeout=60, headers={"User-Agent": "Mozilla/5.0"}).text
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    for a in soup.find_all("a", href=True):
        href = requests.compat.urljoin(page_url, a["href"])
        if re.search(r"\.(zip|csv|gz|xlsx|pdf)(\?|$)", href, re.I) or "siteassets" in href:
            rows.append({"text": " ".join(a.get_text(" ").split()), "url": href})
    return pd.DataFrame(rows).drop_duplicates("url")


def download(url: str, dest: Path, chunk=1 << 22) -> Path:
    dest = Path(dest); dest.parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, timeout=120, headers={"User-Agent": "Mozilla/5.0"}) as r:
        r.raise_for_status()
        total = int(r.headers.get("Content-Length", 0))
        if dest.exists() and total and dest.stat().st_size == total:
            print(f"Already downloaded: {dest.name}")
            return dest
        tmp = dest.with_suffix(dest.suffix + ".part")
        done = 0
        with open(tmp, "wb") as f:
            for b in r.iter_content(chunk):
                f.write(b); done += len(b)
                if total and done % (chunk * 64) < chunk:
                    print(f"  {done/1e9:.2f}/{total/1e9:.2f} GB")
        tmp.rename(dest)
    print(f"Saved {dest} ({dest.stat().st_size/1e6:.1f} MB)")
    return dest


def csv_members(path: Path) -> list[str]:
    """CSV files inside a zip (or the path itself if it is a CSV)."""
    if zipfile.is_zipfile(path):
        return [m for m in zipfile.ZipFile(path).namelist() if m.lower().endswith(".csv")]
    return [str(path)]


def open_csv(path: Path, member: str):
    if zipfile.is_zipfile(path):
        return zipfile.ZipFile(path).open(member)
    return open(member, "rb")


def _find(cols, *pats, exclude=()):
    for p in pats:
        for c in cols:
            lc = c.lower().strip()
            if re.fullmatch(p, lc) and not any(re.search(e, lc) for e in exclude):
                return c
    return None


def detect_drive_cols(cols) -> dict:
    m = {
        "lat": _find(cols, r"lat(itude)?"),
        "lon": _find(cols, r"lon(g|gitude)?"),
        "mcc": _find(cols, r"mcc"),
        "mnc": _find(cols, r"mnc"),

        "rsrp": _find(cols, r"rsrp", r"rsrp[_ ]?(1|0|best|top|max)?", r".*rsrp.*",
                      exclude=(r"rsrq", r"ss_", r"[234]$")),
        "date": _find(cols, r"date", r"month.*"),
    }
    missing = [k for k in ("lat", "lon", "mnc", "rsrp") if m[k] is None]
    if missing:
        raise ValueError(f"Could not detect {missing} in columns: {list(cols)}")
    return m


def aggregate_drive_test(path: Path, chunksize=2_000_000, sample_frac=0.01, seed=0):
    """Stream the drive-test CSV(s) -> grid x operator aggregates (+ small point sample)."""
    from pyproj import Transformer
    tf = Transformer.from_crs(4326, 27700, always_xy=True)
    parts, samples = [], []
    for member in csv_members(path):
        with open_csv(path, member) as fh:
            head = pd.read_csv(fh, nrows=0).columns
        cm = detect_drive_cols(head)
        print(f"{Path(member).name}: using {cm}")
        usecols = [c for c in cm.values() if c]
        with open_csv(path, member) as fh:
            for i, ch in enumerate(pd.read_csv(fh, usecols=usecols, chunksize=chunksize, low_memory=False)):
                ch = ch.rename(columns={v: k for k, v in cm.items() if v})
                for c in ("lat", "lon", "rsrp", "mnc", "mcc"):
                    if c in ch:
                        ch[c] = pd.to_numeric(ch[c], errors="coerce")
                if "mcc" in ch:
                    ch = ch[ch["mcc"].isna() | (ch["mcc"] == 234)]
                ch["operator"] = ch["mnc"].map(UK_MNC)
                ch = ch.dropna(subset=["lat", "lon", "rsrp", "operator"])
                ch = ch[ch["lat"].between(49.8, 61) & ch["lon"].between(-8.7, 2) & ch["rsrp"].between(-160, -30)]
                x, y = tf.transform(ch["lon"].to_numpy(), ch["lat"].to_numpy())
                ch["gx"] = (x // GRID_M).astype("int32"); ch["gy"] = (y // GRID_M).astype("int32")
                ch["cov_out"] = (ch["rsrp"] >= RSRP_OUTDOOR_DBM).astype("int32")
                ch["cov_in"] = (ch["rsrp"] >= RSRP_INDOOR_DBM).astype("int32")
                g = ch.groupby(["gx", "gy", "operator"]).agg(
                    n=("rsrp", "size"), rsrp_sum=("rsrp", "sum"),
                    cov_out=("cov_out", "sum"), cov_in=("cov_in", "sum")).reset_index()
                parts.append(g)
                if sample_frac:
                    samples.append(ch.sample(frac=sample_frac, random_state=seed + i)
                                   [["lat", "lon", "operator", "rsrp"] + (["date"] if "date" in ch else [])])
                print(f"  chunk {i}: {len(ch):,} valid rows")
    grid = (pd.concat(parts).groupby(["gx", "gy", "operator"], as_index=False).sum())
    grid["rsrp_mean"] = grid["rsrp_sum"] / grid["n"]
    grid["cov_out_share"] = grid["cov_out"] / grid["n"]
    grid["cov_in_share"] = grid["cov_in"] / grid["n"]
    pts = pd.concat(samples, ignore_index=True) if samples else None
    return grid, pts


LAD_SERVICES = [
    "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/Local_Authority_Districts_December_2024_Boundaries_UK_BGC/FeatureServer/0",
    "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/LAD_DEC_24_UK_BSC/FeatureServer/0",
    "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/Local_Authority_Districts_December_2024_Boundaries_UK_BSC/FeatureServer/0",
]


def load_lad_boundaries(cache: Path):
    import geopandas as gpd
    cache = Path(cache)
    if cache.exists():
        return gpd.read_file(cache)
    last_err = None
    for svc in LAD_SERVICES:
        try:
            feats, offset = [], 0
            while True:
                r = requests.get(f"{svc}/query", params={
                    "where": "1=1", "outFields": "*", "outSR": 4326, "f": "geojson",
                    "resultOffset": offset, "resultRecordCount": 200}, timeout=120)
                r.raise_for_status()
                js = r.json()
                if "features" not in js:
                    raise RuntimeError(str(js)[:300])
                feats += js["features"]
                if len(js["features"]) < 200:
                    break
                offset += 200
            gdf = gpd.GeoDataFrame.from_features(feats, crs=4326)
            cd = next(c for c in gdf.columns if re.fullmatch(r"lad\d{2}cd", c.lower()))
            nm = next(c for c in gdf.columns if re.fullmatch(r"lad\d{2}nm", c.lower()))
            gdf = gdf.rename(columns={cd: "lad_cd", nm: "lad_nm"})[["lad_cd", "lad_nm", "geometry"]]
            gdf.to_file(cache, driver="GeoJSON")
            print(f"LAD boundaries: {len(gdf)} from {svc.split('/services/')[1]}")
            return gdf
        except Exception as e:
            last_err = e
            print("  failed:", svc.split("/services/")[1], "->", str(e)[:120])
    raise RuntimeError(f"All LAD services failed: {last_err}. Download the GeoJSON manually from geoportal.statistics.gov.uk to {cache}")


def grid_to_lad(grid: pd.DataFrame, lad):
    import geopandas as gpd
    cells = grid[["gx", "gy"]].drop_duplicates()
    pts = gpd.GeoDataFrame(cells, geometry=gpd.points_from_xy((cells.gx + .5) * GRID_M, (cells.gy + .5) * GRID_M), crs=27700)
    j = gpd.sjoin(pts, lad.to_crs(27700)[["lad_cd", "lad_nm", "geometry"]], how="left", predicate="within")

    miss = j["lad_cd"].isna()
    if miss.any():
        near = gpd.sjoin_nearest(pts[miss.values], lad.to_crs(27700)[["lad_cd", "lad_nm", "geometry"]], how="left")
        near = near[~near.index.duplicated()]
        j.loc[near.index, ["lad_cd", "lad_nm"]] = near[["lad_cd", "lad_nm"]].values
    return grid.merge(j[["gx", "gy", "lad_cd", "lad_nm"]].drop_duplicates(["gx", "gy"]), on=["gx", "gy"], how="left")


def grid_gap_table(grid: pd.DataFrame, min_n=20) -> pd.DataFrame:
    """Wide grid table: one row per 1 km cell, operator coverage side by side, Vodafone gap."""
    g = grid[grid["n"] >= min_n]
    wide = g.pivot_table(index=["gx", "gy", "lad_cd", "lad_nm"], columns="operator",
                         values=["cov_out_share", "cov_in_share", "rsrp_mean", "n"])
    wide.columns = [f"{v}_{op}" for v, op in wide.columns]
    wide = wide.reset_index()
    need = [f"cov_out_share_{o}" for o in ("Vodafone", "EE", "O2")]
    wide = wide.dropna(subset=need)
    for m in ("cov_out_share", "cov_in_share", "rsrp_mean"):
        wide[f"best_rival_{m}"] = wide[[f"{m}_EE", f"{m}_O2"]].max(axis=1)
        wide[f"vo_gap_{m}"] = wide[f"best_rival_{m}"] - wide[f"{m}_Vodafone"]
        if f"{m}_Three" in wide:
            wide[f"{m}_VodafoneThree"] = wide[[f"{m}_Vodafone", f"{m}_Three"]].max(axis=1)
    wide["vo_notspot_out"] = 1 - wide["cov_out_share_Vodafone"]
    wide["vo_notspot_in"] = 1 - wide["cov_in_share_Vodafone"]
    return wide


def lad_gap_table(wide: pd.DataFrame, min_cells=5) -> pd.DataFrame:
    metrics = [c for c in wide.columns if c.startswith(("cov_", "rsrp_", "best_rival_", "vo_"))]
    la = wide.groupby(["lad_cd", "lad_nm"])[metrics].mean()
    la["n_cells"] = wide.groupby(["lad_cd", "lad_nm"]).size()
    la["share_cells_vo_behind"] = wide.assign(b=wide["vo_gap_cov_out_share"] > 0.1).groupby(["lad_cd", "lad_nm"])["b"].mean()
    la = la[la["n_cells"] >= min_cells].reset_index()
    return la.sort_values("vo_gap_cov_out_share", ascending=False)


OCID_COLS = ["radio", "mcc", "net", "area", "cell", "unit", "lon", "lat",
             "range", "samples", "changeable", "created", "updated", "averageSignal"]


def load_opencellid(path: Path) -> pd.DataFrame:
    peek = pd.read_csv(path, nrows=1, header=None)
    has_header = str(peek.iloc[0, 0]).lower() == "radio"
    df = pd.read_csv(path, header=0 if has_header else None, names=None if has_header else OCID_COLS, low_memory=False)
    df.columns = [c.strip() for c in df.columns]
    df = df[(df["mcc"] == 234)].copy()
    df["operator"] = df["net"].map(UK_MNC)
    df = df.dropna(subset=["operator", "lat", "lon"])
    df["updated_dt"] = pd.to_datetime(df["updated"], unit="s", errors="coerce")
    return df


def opencellid_by_lad(cells: pd.DataFrame, lad, radios=("LTE", "NR"), site_round=3) -> pd.DataFrame:
    """Cells and approximate sites (cells snapped to ~100 m) per LA x operator."""
    import geopandas as gpd
    c = cells[cells["radio"].isin(radios)].copy()
    c["site"] = c["lat"].round(site_round).astype(str) + "_" + c["lon"].round(site_round).astype(str)
    g = gpd.GeoDataFrame(c, geometry=gpd.points_from_xy(c.lon, c.lat), crs=4326)
    j = gpd.sjoin(g, lad[["lad_cd", "geometry"]], how="inner", predicate="within")
    out = j.groupby(["lad_cd", "operator"]).agg(cells=("cell", "size"), sites=("site", "nunique")).reset_index()
    wide = out.pivot(index="lad_cd", columns="operator", values=["cells", "sites"]).fillna(0)
    wide.columns = [f"ocid_{v}_{op}" for v, op in wide.columns]
    area = lad.to_crs(27700).set_index("lad_cd").area / 1e6
    wide = wide.join(area.rename("area_km2"), how="right").fillna(0)
    for op in OPS:
        if f"ocid_sites_{op}" in wide:
            wide[f"ocid_site_density_{op}"] = wide[f"ocid_sites_{op}"] / wide["area_km2"]
    if {"ocid_sites_EE", "ocid_sites_O2", "ocid_sites_Vodafone"} <= set(wide.columns):
        wide["ocid_vo_site_share"] = wide["ocid_sites_Vodafone"] / wide[
            ["ocid_sites_EE", "ocid_sites_O2", "ocid_sites_Vodafone"]].sum(axis=1).replace(0, np.nan)
    return wide.reset_index()


Overwriting /content/collect_lib.py


In [13]:
import sys, importlib; sys.path.insert(0, '/content')
import collect_lib as cl; importlib.reload(cl)
import pandas as pd, numpy as np
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)

## 2. Ofcom drive-test data: find and download


In [14]:
DT_URLS = ['https://static.ofcom.org.uk/static/research/mobile-signal-strength-data/4g-lte-2023-mobile-signal-measurement-data.zip']
print(DT_URLS)

['https://static.ofcom.org.uk/static/research/mobile-signal-strength-data/4g-lte-2023-mobile-signal-measurement-data.zip']


In [15]:
dt_files = []
for u in DT_URLS:
    name = u.split('/')[-1].split('?')[0] or 'drive_test.zip'
    dt_files.append(cl.download(u, RAW / name))
for f in dt_files:
    print(f.name, '->', cl.csv_members(f)[:10])

Already downloaded: 4g-lte-2023-mobile-signal-measurement-data.zip
4g-lte-2023-mobile-signal-measurement-data.zip -> ['lte_2023.csv']


### 2b. Peek at the raw file


In [20]:
%%writefile -a /content/collect_lib.py

def _suffix_to_op(s):
    s = s.lower()
    if s in ("3uk", "h3g", "three", "3", "h3"): return "Three"
    if s in ("ee", "eeuk"): return "EE"
    if s in ("o2", "o2uk", "tf", "vmo2"): return "O2"
    if s.startswith(("vf", "vod", "vo")): return "Vodafone"
    return None

def detect_wide_cols(cols):
    lat = _find(cols, r"lat(itude)?"); lon = _find(cols, r"lon(g|gitude)?")
    date = _find(cols, r"month_year", r"date", r"month.*")
    ops = {}
    for c in cols:
        m = re.fullmatch(r"rsrp_top1_(.+)", c.lower().strip())
        if m and _suffix_to_op(m.group(1)):
            ops[_suffix_to_op(m.group(1))] = c
    if not (lat and lon and ops):
        raise ValueError(f"Wide format not recognised. Columns: {list(cols)[:40]}")
    unknown = [c for c in cols if re.fullmatch(r"rsrp_top1_(.+)", c.lower()) and c not in ops.values()]
    return {"lat": lat, "lon": lon, "date": date, "ops": ops, "unmapped_rsrp_cols": unknown}

def aggregate_drive_test(path, chunksize=1_000_000, sample_frac=0.01, seed=0):
    from pyproj import Transformer
    tf = Transformer.from_crs(4326, 27700, always_xy=True)
    parts, samples = [], []
    for member in csv_members(path):
        with open_csv(path, member) as fh:
            head = pd.read_csv(fh, nrows=0).columns
        cm = detect_wide_cols(head)
        print(f"{Path(member).name}: operators {cm['ops']}  unmapped={cm['unmapped_rsrp_cols']}")
        usecols = [cm["lat"], cm["lon"]] + ([cm["date"]] if cm["date"] else []) + list(cm["ops"].values())
        with open_csv(path, member) as fh:
            for i, ch in enumerate(pd.read_csv(fh, usecols=usecols, chunksize=chunksize, low_memory=False)):
                ch = ch.rename(columns={cm["lat"]: "lat", cm["lon"]: "lon"})
                ch["lat"] = pd.to_numeric(ch["lat"], errors="coerce"); ch["lon"] = pd.to_numeric(ch["lon"], errors="coerce")
                ch = ch[ch["lat"].between(49.8, 61) & ch["lon"].between(-8.7, 2)]
                R = pd.DataFrame({op: pd.to_numeric(ch[c], errors="coerce") for op, c in cm["ops"].items()}, index=ch.index)
                R = R.where((R >= -160) & (R <= -30))
                keep = R.notna().any(axis=1)
                ch, R = ch[keep], R[keep]
                x, y = tf.transform(ch["lon"].to_numpy(), ch["lat"].to_numpy())
                gx = (x // GRID_M).astype("int32"); gy = (y // GRID_M).astype("int32")
                for op in R.columns:
                    r = R[op].to_numpy()
                    d = pd.DataFrame({"gx": gx, "gy": gy, "operator": op, "n": 1,
                                      "n_sig": ~np.isnan(r), "rsrp_sum": np.nan_to_num(r),
                                      "cov_out": np.nan_to_num(r, nan=-999) >= RSRP_OUTDOOR_DBM,
                                      "cov_in": np.nan_to_num(r, nan=-999) >= RSRP_INDOOR_DBM})
                    parts.append(d.groupby(["gx", "gy", "operator"], as_index=False)[["n", "n_sig", "rsrp_sum", "cov_out", "cov_in"]].sum())
                if sample_frac:
                    s = ch.join(R.add_prefix("rsrp_")).sample(frac=sample_frac, random_state=seed + i)
                    samples.append(s.drop(columns=list(cm["ops"].values()), errors="ignore"))
                print(f"  chunk {i}: {len(ch):,} rows")
    grid = pd.concat(parts).groupby(["gx", "gy", "operator"], as_index=False).sum()
    grid["rsrp_mean"] = grid["rsrp_sum"] / grid["n_sig"].replace(0, np.nan)
    grid["cov_out_share"] = grid["cov_out"] / grid["n"]
    grid["cov_in_share"] = grid["cov_in"] / grid["n"]
    grid["signal_seen_share"] = grid["n_sig"] / grid["n"]
    pts = pd.concat(samples, ignore_index=True) if samples else None
    return grid, pts

Appending to /content/collect_lib.py


In [21]:
import importlib; importlib.reload(cl)
with cl.open_csv(dt_files[0], cl.csv_members(dt_files[0])[0]) as fh:
    print(cl.detect_wide_cols(pd.read_csv(fh, nrows=0).columns))

GRID_FILE = OUT / 'drivetest_grid_1km.parquet'
PTS_FILE  = OUT / 'drivetest_points_sample.parquet'
parts, samples = [], []
for f in dt_files:
    g, p = cl.aggregate_drive_test(f)
    parts.append(g); samples.append(p)
grid = pd.concat(parts).groupby(['gx','gy','operator'], as_index=False)[['n','n_sig','rsrp_sum','cov_out','cov_in']].sum()
grid['rsrp_mean'] = grid.rsrp_sum / grid.n_sig.replace(0, np.nan)
grid['cov_out_share'] = grid.cov_out / grid.n
grid['cov_in_share'] = grid.cov_in / grid.n
grid.to_parquet(GRID_FILE); pd.concat(samples).to_parquet(PTS_FILE)
print(grid.shape)
grid.groupby('operator')[['n','n_sig']].sum()

{'lat': 'latitude', 'lon': 'longitude', 'date': 'month_year', 'ops': {'Three': 'rsrp_top1_3uk', 'EE': 'rsrp_top1_ee', 'O2': 'rsrp_top1_o2', 'Vodafone': 'rsrp_top1_vf'}, 'unmapped_rsrp_cols': []}
lte_2023.csv: operators {'Three': 'rsrp_top1_3uk', 'EE': 'rsrp_top1_ee', 'O2': 'rsrp_top1_o2', 'Vodafone': 'rsrp_top1_vf'}  unmapped=[]
  chunk 0: 881,890 rows
  chunk 1: 883,461 rows
  chunk 2: 880,676 rows
  chunk 3: 882,705 rows
  chunk 4: 880,747 rows
  chunk 5: 880,378 rows
  chunk 6: 879,460 rows
  chunk 7: 882,821 rows
  chunk 8: 884,449 rows
  chunk 9: 884,555 rows
  chunk 10: 883,066 rows
  chunk 11: 881,423 rows
  chunk 12: 429,262 rows
(51680, 11)


,n,n_sig
operator,,
EE,11014893,10905566
O2,11014893,10851999
Three,11014893,10746596
Vodafone,11014893,10867308


## 3. Local authority boundaries (ONS) and the 1 km grid joined to LAs

In [22]:
lad = cl.load_lad_boundaries(OUT / 'lad_dec2024_uk.geojson')
print(len(lad), 'LAs'); lad.head()

LAD boundaries: 361 from Local_Authority_Districts_December_2024_Boundaries_UK_BGC/FeatureServer/0
361 LAs


,lad_cd,lad_nm,geometry
0,E06000001,Hartlepool,"MULTIPOLYGON (((-1.23001 54.62512, -1.2299 54...."
1,E06000002,Middlesbrough,"MULTIPOLYGON (((-1.27712 54.54791, -1.2772 54...."
2,E06000003,Redcar and Cleveland,"MULTIPOLYGON (((-1.20098 54.57763, -1.20037 54..."
3,E06000004,Stockton-on-Tees,"MULTIPOLYGON (((-1.27493 54.55187, -1.27546 54..."
4,E06000005,Darlington,"POLYGON ((-1.638 54.6172, -1.63767 54.6167, -1..."


In [23]:
grid = cl.grid_to_lad(grid, lad)
wide = cl.grid_gap_table(grid, min_n=20)
la_dt = cl.lad_gap_table(wide, min_cells=5)
wide.to_parquet(OUT / 'grid_gap_1km.parquet')
la_dt.to_csv(OUT / 'la_drivetest_gap.csv', index=False)
print(f'{len(wide):,} head-to-head grid cells | {len(la_dt)} LAs with >=5 cells')
la_dt[['lad_nm','n_cells','cov_out_share_Vodafone','best_rival_cov_out_share','vo_gap_cov_out_share','share_cells_vo_behind']].head(15).round(3)

12,626 head-to-head grid cells | 160 LAs with >=5 cells


,lad_nm,n_cells,cov_out_share_Vodafone,best_rival_cov_out_share,vo_gap_cov_out_share,share_cells_vo_behind
83,Lichfield,20,0.761,0.960,0.199,0.350
143,Argyll and Bute,436,0.707,0.895,0.188,0.349
5,East Riding of Yorkshire,11,0.836,1.000,0.164,0.273
135,Highland,1588,0.807,0.953,0.147,0.247
158,Rhondda Cynon Taf,18,0.800,0.937,0.137,0.278
93,Malvern Hills,68,0.807,0.932,0.125,0.353
52,Basingstoke and Deane,31,0.803,0.925,0.122,0.387
140,Stirling,409,0.843,0.955,0.112,0.215
137,Moray,163,0.861,0.972,0.111,0.252
44,Teignbridge,17,0.895,1.000,0.105,0.176


## 4. OpenCelliD: cell sites per operator per LA

In [24]:
from google.colab import userdata
import requests
OC_FILE = RAW / 'opencellid_234.csv.gz'
if not OC_FILE.exists():
    tok = userdata.get('OPENCELLID_TOKEN')
    r = requests.get('https://opencellid.org/ocid/downloads', params={'token': tok, 'type': 'mcc', 'file': '234.csv.gz'}, timeout=300)
    if r.status_code != 200 or r.content[:2] != b'\x1f\x8b':
        raise RuntimeError(f'Download failed ({r.status_code}): {r.text[:300]}  (token wrong, or daily download limit hit; try tomorrow)')
    OC_FILE.write_bytes(r.content)
cells = cl.load_opencellid(OC_FILE)
print(f'{len(cells):,} UK cells'); display(cells.groupby(['operator','radio']).size().unstack(fill_value=0))

146,428 UK cells


radio,GSM,LTE,NR,UMTS
operator,,,,
EE,3949,31158,2336,176
O2,2703,30150,1409,4305
Three,5,20837,476,1414
Vodafone,6489,39360,1295,366


In [25]:
la_oc = cl.opencellid_by_lad(cells, lad)
la_oc.to_csv(OUT / 'la_opencellid.csv', index=False)
la_oc.sort_values('ocid_vo_site_share').head(10).round(3)

,lad_cd,ocid_cells_EE,ocid_cells_O2,ocid_cells_Three,ocid_cells_Vodafone,ocid_sites_EE,ocid_sites_O2,ocid_sites_Three,ocid_sites_Vodafone,area_km2,ocid_site_density_EE,ocid_site_density_O2,ocid_site_density_Three,ocid_site_density_Vodafone,ocid_vo_site_share
330,S12000039,45.0,0.0,16.0,0.0,44.0,0.0,16.0,0.0,177.287,0.248,0.000,0.090,0.0,0.0
315,S12000018,55.0,23.0,0.0,0.0,49.0,19.0,0.0,0.0,161.910,0.303,0.117,0.000,0.0,0.0
312,S12000013,2.0,0.0,102.0,0.0,2.0,0.0,102.0,0.0,3093.175,0.001,0.000,0.033,0.0,0.0
334,S12000045,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,174.478,0.006,0.000,0.000,0.0,0.0
297,N09000002,53.0,39.0,28.0,0.0,52.0,38.0,23.0,0.0,1437.433,0.036,0.026,0.016,0.0,0.0
302,N09000007,44.0,27.0,27.0,0.0,44.0,27.0,27.0,0.0,510.566,0.086,0.053,0.053,0.0,0.0
305,N09000010,28.0,3.0,0.0,0.0,27.0,3.0,0.0,0.0,1682.845,0.016,0.002,0.000,0.0,0.0
303,N09000008,9.0,0.0,0.0,0.0,9.0,0.0,0.0,0.0,1061.642,0.008,0.000,0.000,0.0,0.0
82,E07000046,7.0,23.0,0.0,0.0,7.0,23.0,0.0,0.0,985.295,0.007,0.023,0.000,0.0,0.0
110,E07000087,11.0,17.0,20.0,0.0,11.0,14.0,12.0,0.0,74.249,0.148,0.189,0.162,0.0,0.0


## 5. Ofcom Connected Nations 2025 (validation)

In [27]:
from google.colab import files
import zipfile, io
up = files.upload()                      # pick the zip you just downloaded
name, data = next(iter(up.items()))
z = zipfile.ZipFile(io.BytesIO(data)); print(z.namelist())
m = next(n for n in z.namelist() if 'laua' in n.lower())
cn = pd.read_csv(z.open(m)); cn.to_csv(OUT / 'cn2025_mobile_laua.csv', index=False)
print(cn.shape); cn.iloc[:3, :12]

Saving 202507_mobile_coverage_r01.zip to 202507_mobile_coverage_r01.zip
['202507_mobile_coverage_r01/', '202507_mobile_coverage_r01/202507_mobile_coverage_4G5G_r01.csv', '202507_mobile_coverage_r01/202507_mobile_coverage_devcon_r01.csv', '202507_mobile_coverage_r01/202507_mobile_coverage_laua_r01.csv', '202507_mobile_coverage_r01/202507_mobile_coverage_pcon_r01.csv', '202507_mobile_coverage_r01/202507_mobile_coverage_UK_and_nations_r01.csv']
(361, 161)


,laua,laua_name,prem_count,pixel_count,ab_rd_count,mway_count,mway_ard_count,2G_prem_out_0,2G_prem_out_1,2G_prem_out_2,2G_prem_out_3,2G_prem_in_0
0,S12000033,Aberdeen City,131903,18582,1624,NaN,1229,NaN,0.01,1.75,98.24,0.13
1,S12000034,Aberdeenshire,130651,631855,20865,NaN,11207,0.63,1.21,12.27,85.88,4.90
2,E07000223,Adur,30319,4220,349,NaN,330,NaN,0.00,0.62,99.38,0.01


## 6. Master LA table

In [28]:
code_col = next(c for c in cn.columns if c.lower() in ('laua','lad_cd','la_code'))
keep_cn = [code_col] + [c for c in cn.columns if c.lower().startswith(('4g_prem_in','4g_prem_out','4g_geo_out','prem_count'))]
master = (lad[['lad_cd','lad_nm']]
          .merge(la_oc, on='lad_cd', how='left')
          .merge(la_dt.drop(columns='lad_nm'), on='lad_cd', how='left')
          .merge(cn[keep_cn].rename(columns={code_col:'lad_cd'}), on='lad_cd', how='left'))
master.to_csv(OUT / 'la_master_stage1.csv', index=False)
print(master.shape)
print('LAs with drive-test gap metric:', master.vo_gap_cov_out_share.notna().sum())
master[['lad_nm','area_km2','ocid_vo_site_share','vo_gap_cov_out_share','cov_out_share_Vodafone','cov_out_share_VodafoneThree']].sort_values('vo_gap_cov_out_share', ascending=False).head(15).round(3)

(361, 57)
LAs with drive-test gap metric: 160


,lad_nm,area_km2,ocid_vo_site_share,vo_gap_cov_out_share,cov_out_share_Vodafone,cov_out_share_VodafoneThree
183,Lichfield,331.216,0.251,0.199,0.761,0.943
327,Argyll and Bute,7004.010,0.067,0.188,0.707,0.755
10,East Riding of Yorkshire,2404.905,0.221,0.164,0.836,0.967
314,Highland,26157.103,0.287,0.147,0.807,0.864
353,Rhondda Cynon Taf,424.147,0.165,0.137,0.800,0.915
216,Malvern Hills,577.126,0.299,0.125,0.807,0.927
107,Basingstoke and Deane,633.792,0.280,0.122,0.803,0.876
324,Stirling,2252.598,0.373,0.112,0.843,0.899
317,Moray,2237.719,0.890,0.111,0.861,0.900
81,Teignbridge,673.735,0.629,0.105,0.895,0.906


In [29]:
from google.colab import files
files.download(str(OUT / 'la_master_stage1.csv'))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>